<a href="https://colab.research.google.com/github/RashmiBansode2002/Conversion-of-Text-to-Gloss-for-Sign-Language/blob/main/English_sentences_extraction.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# <center>Script for extracting the English sentences</center>

---

In [ ]:
#imports
from bs4 import BeautifulSoup
import requests
from urllib.parse import urljoin
import urllib.request
import pandas as pd
import pickle

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
#url of the ISL Corpus
url_isl_corpus = "https://www.sign-lang.uni-hamburg.de/meinedgs/ling/start-name_en.html"

#request the ISL corpus page
r = requests.get(url_isl_corpus)

#get the html contennt of the ISL corpus page
html = r.text

In [ ]:
#create a content soup from the html content of the dgs corpus page with BeautifulSoup
content_soup = BeautifulSoup(html, 'html.parser')

In [ ]:
#rows with all types of files - ILEX, EAF, MP4...
rows_with_transcripts = content_soup.find('table', {'class': 'transcripts'}).find_all('tr')

In [ ]:
#list with all hrefs of the EAF files
list_eaf_files = []

In [ ]:
#get all the cells with transcripts data
for r in rows_with_transcripts[1:]:
    cells_with_transcripts = r.find_all('td')

    #cells with the EAF transcript files
    eaf_files = cells_with_transcripts[5]

    #add the href of each EAF transcript file to a list
    if(eaf_files.find('a')) != None:
        list_eaf_files.append(eaf_files.find('a').attrs['href'])

In [ ]:
#list with the absolute urls of each EAF transcript
absolute_paths_eaf_transcripts = []

In [ ]:
#create an absolute path for each EAF transcript with
#taking the base url of the DGS Corpus and
#the href of each EAF transcript from the list_eaf_files

for single_eaf in list_eaf_files:
    absolute_url = urljoin(url_isl_corpus, single_eaf)
    absolute_paths_eaf_transcripts.append(absolute_url)

In [ ]:
#this is how an element from the list looks like:
absolute_paths_eaf_transcripts[0]

'https://www.sign-lang.uni-hamburg.de/meinedgs/eaf/1413451-11105600-11163240.eaf'

<hr style="border:1.5px solid gray"></hr>

#### Speakers A:

  - *English sentences:*
       - <b>TIER_ID="What is your name?"</b>
  - *English gloss sentences:*
       - <b>TIER_ID="Your name what?"

<hr style="border: 0.5px solid gray"></hr>   

#### Speakers B:
  - *English sentences:*
       - <b>My name is Aditi</b>
  - *English gloss sentences:*
       - <b>Aditi my name</b>

<hr style="border:1.5px solid gray"></hr>

### <center>Extract the English sentences from speakers A <center>     

In [ ]:
#this is a list with the content of all tags that have the attribute "ANNOTATION_VALUE" (they include english glosses, english sentences, etc.)

#from this content *only* the tags with english sentences from speakers A must be extracted
transcript_content_a = []

#this is a list for the specific time encoding of each sentence
time_encodings_a = []

#this is a list of the english sentences
english_sentences_a = []

In [ ]:
%%time
#take each transcript from the list with all EAF transcripts and read its content to extract
#in a data frame all english sentences *only* for speakers A
for transcript in absolute_paths_eaf_transcripts:
    with urllib.request.urlopen(transcript) as f:
        content = f.read().decode('utf-8')
        transcript_content_a = BeautifulSoup(content, 'xml').find_all(name="ANNOTATION_VALUE")
        time_encodings_a = BeautifulSoup(content, 'xml').find_all(name="TIME_SLOT")
        for value in range(0, len(transcript_content_a)):
            #if the value of the tags attribute TIER_ID is a english sentence from speaker A, extract it
            if transcript_content_a[value].parent.parent.parent.attrs['TIER_ID'] == "Deutsche_Übersetzung_A":
                #this is the time encoding for the sentence (both starting and ending)
                time = transcript_content_a[value].parent.attrs
                #this is the starting time of the sentence
                start = time['TIME_SLOT_REF1']
                #this is the ending time of the sentencec
                end = time['TIME_SLOT_REF2']
                #the sentence itself
                sentence_a = transcript_content_a[value].text
                #group the sentence + its start + its end
                sentence_group_a = [sentence_a, start, end]
                #add the english sentence to the list of english sentences
                english_sentences_a.append(sentence_group_a)

CPU times: user 6min 2s, sys: 3.66 s, total: 6min 5s
Wall time: 10min 46s


#### Save the list with English sentences A using pickle

In [ ]:
#save the list using pickle where each element is: sentence A + start + end
#to be used later without extracting it again

with open("path", "wb") as fp:
    pickle.dump(english_sentences_a, fp)

#### Create a data frame for the sentences A

In [ ]:
#list with only the sentence (no timestamps included)
data_a = []

for s in range(0, len(english_sentences_a)):
    data_a.append(english_sentences_a[s][0])

In [ ]:
df_a = pd.read_csv("/content/drive/MyDrive/text-to-gloss-sign-language-translation-main/data/gloss 1 train.csv")

print("Number of rows:", df_a.shape[0])
print("Number of columns:", df_a.shape[1])

# Display all the data in the DataFrame
print(df_a)

Number of rows: 84
Number of columns: 1
                 gloss
0              I angry
1            you angry
2        angry you me 
3        you angry not
4         I angry very
..                 ...
79            you cry?
80  promise you sleep?
81           thank you
82            promise 
83   I want night food

[84 rows x 1 columns]


In [ ]:
#save it as a file where each sentence is on a new line
df_a.to_csv(f"path", encoding="utf-8-sig", index=False, header=False)

<hr style="border:1.5px solid gray"></hr>

### <center>Extract the english sentences from the transcripts B<center>  

In [ ]:
#this is a list with the content of all tags that have the attribute "ANNOTATION_VALUE" (they include english glosses, english sentences,
#english glosses, english sentences, etc.)

#from this content *only* the tags with english sentences from speakers B must be extracted
transcript_content_b = []

#this is a list for the specific time encoding of each sentence
time_encodings_b = []

#this is a list of the english sentences
english_sentences_b = []

In [ ]:
%%time
#take each transcript from the list with all EAF transcripts and read its content to extract
#in a data frame all english sentences *only* for speakers B
for transcript in absolute_paths_eaf_transcripts:
    with urllib.request.urlopen(transcript) as f:
        content = f.read().decode('utf-8')
        transcript_content_b = BeautifulSoup(content, 'xml').find_all(name="ANNOTATION_VALUE")
        time_encodings_b = BeautifulSoup(content, 'xml').find_all(name="TIME_SLOT")
        for value in range(0, len(transcript_content_b)):
            #if the value of the tags attribute TIER_ID is a english sentence from speaker B, extract it
            if transcript_content_b[value].parent.parent.parent.attrs['TIER_ID'] == "Deutsche_Übersetzung_B":
                #this is the time encoding for the sentence (both starting and ending)
                time = transcript_content_b[value].parent.attrs
                #this is the starting time of the sentence
                start = time['TIME_SLOT_REF1']
                #this is the ending time of the sentencec
                end = time['TIME_SLOT_REF2']
                #the sentence itself
                sentence_b = transcript_content_b[value].text
                #group the sentence + its start + its end
                sentence_group_b = [sentence_b, start, end]
                #add the english sentence to the list of english sentences
                english_sentences_b.append(sentence_group_b)

CPU times: user 6min 14s, sys: 3.22 s, total: 6min 18s
Wall time: 11min 1s


[link text](https://)#### Save the list with english sentences B using pickle

In [ ]:
#save the list using pickle where each element is: sentence + start + time
#to use it for extracting the gloss sentences later

with open("path", "wb") as fp:
    pickle.dump(english_sentences_b, fp)

#### Create a data frame for the sentences B

In [ ]:
#list with only the sentence from the list with sentences B (no timestamps included)
data_b = []

for s in range(0, len(english_sentences_b)):
    data_b.append(english_sentences_b[s][0])

<hr style="border:1.5px solid gray"></hr>